### Preprocessing

Cleans each participant's data and cuts it into 1-second windows with summary features.

**Input:** raw CSV files in `data/raw/`
**Output:** one table of windows (one row per window) for modelling

#### To do List

**Cleaning**
- [ ] Keep only timestamp, the six sensor columns and label (drops extra index columns)
- [ ] Convert timestamp to datetime
- [ ] Add subject column
- [ ] Map labels to the 8 classes (merge stairs and active cycling; drop inactive cycling)
- [ ] Check label counts and rows dropped before vs after

**Windowing**
- [ ] Find gaps (after dropping rows) and split the recording into continuous stretches
- [ ] Cut each stretch into 1-second windows (50 samples)
- [ ] Decide how to label windows (majority activity, or drop mixed windows)
- [ ] Calculate features per window (mean, standard deviation, range of each axis and vector magnitude)
- [ ] Check the windows table makes sense (about one row per second, sensible labels)

**Scale up**
- [ ] Turn the cleaning and windowing steps into functions in src/
- [ ] Resample S006 from 100 Hz to 50 Hz
- [ ] Run on all 22 participants and combine into one windows table
- [ ] Check total windows and class counts against the audit

**Document**
- [ ] Record decisions in decision_log.md (window labelling, gap handling, resampling)
- [ ] Commit after each section

In [73]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [74]:
import sys
print(sys.executable)
sys.path.append("..")

C:\Users\sarab\Projects\CT4101 ML Project\Machine_Learning_Portfolio_Project\.venv\Scripts\python.exe


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
from pathlib import Path
print(Path.cwd())
# Create data directory 
data_directory = Path("../data/raw")
print(list(data_directory.iterdir()))

C:\Users\sarab\Projects\CT4101 ML Project\Machine_Learning_Portfolio_Project\notebooks
[WindowsPath('../data/raw/S006.csv'), WindowsPath('../data/raw/S008.csv'), WindowsPath('../data/raw/S009.csv'), WindowsPath('../data/raw/S010.csv'), WindowsPath('../data/raw/S012.csv'), WindowsPath('../data/raw/S013.csv'), WindowsPath('../data/raw/S014.csv'), WindowsPath('../data/raw/S015.csv'), WindowsPath('../data/raw/S016.csv'), WindowsPath('../data/raw/S017.csv'), WindowsPath('../data/raw/S018.csv'), WindowsPath('../data/raw/S019.csv'), WindowsPath('../data/raw/S020.csv'), WindowsPath('../data/raw/S021.csv'), WindowsPath('../data/raw/S022.csv'), WindowsPath('../data/raw/S023.csv'), WindowsPath('../data/raw/S024.csv'), WindowsPath('../data/raw/S025.csv'), WindowsPath('../data/raw/S026.csv'), WindowsPath('../data/raw/S027.csv'), WindowsPath('../data/raw/S028.csv'), WindowsPath('../data/raw/S029.csv')]


In [5]:
# Read one csv file from the HARTH dataset 
df = pd.read_csv(data_directory / "S015.csv")

In [6]:
df.head()

,timestamp,index,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label
0,2019-01-12 00:00:00.000,0,-0.999492,0.023192,0.025461,-0.935965,0.023979,-0.355521,6
1,2019-01-12 00:00:00.020,1,-0.995551,0.021414,0.042297,-0.928875,0.029540,-0.371151,6
2,2019-01-12 00:00:00.040,2,-0.989507,0.012187,0.038830,-0.942907,0.019677,-0.359649,6
3,2019-01-12 00:00:00.060,3,-0.991327,0.015926,0.056181,-0.942363,0.009682,-0.348916,6
4,2019-01-12 00:00:00.080,4,-0.997368,0.014239,0.035168,-0.930852,0.012002,-0.357421,6


In [7]:
# Select columns to keep
sensor_cols = ["back_x", "back_y", "back_z", "thigh_x", "thigh_y", "thigh_z"]

data = df[["timestamp"] + sensor_cols + ["label"]]
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 418392 entries, 0 to 418391
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   timestamp  418392 non-null  str    
 1   back_x     418392 non-null  float64
 2   back_y     418392 non-null  float64
 3   back_z     418392 non-null  float64
 4   thigh_x    418392 non-null  float64
 5   thigh_y    418392 non-null  float64
 6   thigh_z    418392 non-null  float64
 7   label      418392 non-null  int64  
dtypes: float64(6), int64(1), str(1)
memory usage: 25.5 MB


In [8]:
# Convert Time from str to datetime64
data["timestamp"] = pd.to_datetime(data["timestamp"])
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 418392 entries, 0 to 418391
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  418392 non-null  datetime64[us]
 1   back_x     418392 non-null  float64       
 2   back_y     418392 non-null  float64       
 3   back_z     418392 non-null  float64       
 4   thigh_x    418392 non-null  float64       
 5   thigh_y    418392 non-null  float64       
 6   thigh_z    418392 non-null  float64       
 7   label      418392 non-null  int64         
dtypes: datetime64[us](1), float64(6), int64(1)
memory usage: 25.5 MB


In [9]:
# Add subject column
data["subject"] = "S015"
data.head()

,timestamp,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label,subject
0,2019-01-12 00:00:00.000,-0.999492,0.023192,0.025461,-0.935965,0.023979,-0.355521,6,S015
1,2019-01-12 00:00:00.020,-0.995551,0.021414,0.042297,-0.928875,0.029540,-0.371151,6,S015
2,2019-01-12 00:00:00.040,-0.989507,0.012187,0.038830,-0.942907,0.019677,-0.359649,6,S015
3,2019-01-12 00:00:00.060,-0.991327,0.015926,0.056181,-0.942363,0.009682,-0.348916,6,S015
4,2019-01-12 00:00:00.080,-0.997368,0.014239,0.035168,-0.930852,0.012002,-0.357421,6,S015


In [10]:
# Activity code label dictionary

label_dict = {1: "walking", 
              2: "running", 
              3: "shuffling", 
              4: "stairs", # changed from stairs (ascending)
              5: "stairs", # changed from stairs (descending)
              6: "standing", 
              7: "sitting", 
              8: "lying",
              13: "cycling", # changed from cycling (sit)
              14: "cycling"} # changed from cycling (stand)
            
# Left out 130: "cycling (sit, inactive)" & 140: "cycling (stand, inactive)"

# Map number to label
data["activity"] = data["label"].map(label_dict)
data.info()
data.head()
print(data.loc[data["label"].isin([130, 140])]) # No inactive cycling data
print(data.loc[data["label"].isin([13,14])])
print(data.loc[data["label"].isin([4,5])])

<class 'pandas.DataFrame'>
RangeIndex: 418392 entries, 0 to 418391
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  418392 non-null  datetime64[us]
 1   back_x     418392 non-null  float64       
 2   back_y     418392 non-null  float64       
 3   back_z     418392 non-null  float64       
 4   thigh_x    418392 non-null  float64       
 5   thigh_y    418392 non-null  float64       
 6   thigh_z    418392 non-null  float64       
 7   label      418392 non-null  int64         
 8   subject    418392 non-null  str           
 9   activity   418392 non-null  str           
dtypes: datetime64[us](1), float64(6), int64(1), str(2)
memory usage: 31.9 MB
Empty DataFrame
Columns: [timestamp, back_x, back_y, back_z, thigh_x, thigh_y, thigh_z, label, subject, activity]
Index: []
                     timestamp    back_x    back_y    back_z   thigh_x  \
397376 2019-01-12 02:14:20.320 -0.886503  0.1611

In [11]:
# Drops inactive cycling columns (this participant doesn't have any)
data = data.dropna(subset=["activity"])
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 418392 entries, 0 to 418391
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  418392 non-null  datetime64[us]
 1   back_x     418392 non-null  float64       
 2   back_y     418392 non-null  float64       
 3   back_z     418392 non-null  float64       
 4   thigh_x    418392 non-null  float64       
 5   thigh_y    418392 non-null  float64       
 6   thigh_z    418392 non-null  float64       
 7   label      418392 non-null  int64         
 8   subject    418392 non-null  str           
 9   activity   418392 non-null  str           
dtypes: datetime64[us](1), float64(6), int64(1), str(2)
memory usage: 31.9 MB


#### Note that dropping inactive cycling rows creates more gaps in the timestamps - remember to consider when creating windows

### Creating Windows for one participant (S015) first
- Sensor data was sampled at 50Hz i.e. the time difference between each row of data is 20 milliseconds 
- To create one second windows we need to combine 50 rows
- the 50 rows need to be continuous so whenever there is a longer gap between 2 rows, the window needs to end and a new window needs to start
- Need to decide whether to discard shorter windows or not

In [12]:
time_between_rows = data["timestamp"].diff() # Time between each row
step = time_between_rows.median() # Median time between 2 rows for a participant

# Create a new block each time the time between rows is above the median time (20ms)
new_block = time_between_rows > step # True = gap
data["block"] = new_block.cumsum() # cumsum() computes the cumulative sum of elements in an array (cumulitive sum of gaps)
data.head()

,timestamp,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label,subject,activity,block
0,2019-01-12 00:00:00.000,-0.999492,0.023192,0.025461,-0.935965,0.023979,-0.355521,6,S015,standing,0
1,2019-01-12 00:00:00.020,-0.995551,0.021414,0.042297,-0.928875,0.029540,-0.371151,6,S015,standing,0
2,2019-01-12 00:00:00.040,-0.989507,0.012187,0.038830,-0.942907,0.019677,-0.359649,6,S015,standing,0
3,2019-01-12 00:00:00.060,-0.991327,0.015926,0.056181,-0.942363,0.009682,-0.348916,6,S015,standing,0
4,2019-01-12 00:00:00.080,-0.997368,0.014239,0.035168,-0.930852,0.012002,-0.357421,6,S015,standing,0


In [13]:
# Check against data inspection
data["block"].nunique() # Should be 1 more than in the Steps >20ms column in the data inspection summary table

37

In [14]:
# Counts the rows in each block and create a window every 50 rows
# Once the block ends move onto the next and start from 0
data["window"] = data.groupby("block").cumcount() // 50
data.head(300000)

,timestamp,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label,subject,activity,block,window
0,2019-01-12 00:00:00.000,-0.999492,0.023192,0.025461,-0.935965,0.023979,-0.355521,6,S015,standing,0,0
1,2019-01-12 00:00:00.020,-0.995551,0.021414,0.042297,-0.928875,0.029540,-0.371151,6,S015,standing,0,0
2,2019-01-12 00:00:00.040,-0.989507,0.012187,0.038830,-0.942907,0.019677,-0.359649,6,S015,standing,0,0
3,2019-01-12 00:00:00.060,-0.991327,0.015926,0.056181,-0.942363,0.009682,-0.348916,6,S015,standing,0,0
4,2019-01-12 00:00:00.080,-0.997368,0.014239,0.035168,-0.930852,0.012002,-0.357421,6,S015,standing,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,2019-01-12 01:41:03.980,-0.898697,0.037736,-0.436966,-0.261783,-0.017452,0.964167,7,S015,sitting,26,1222
299996,2019-01-12 01:41:04.000,-0.893427,0.033640,-0.443220,-0.269176,-0.020524,0.964057,7,S015,sitting,26,1222
299997,2019-01-12 01:41:04.020,-0.895174,0.034239,-0.436369,-0.280004,0.002817,0.964163,7,S015,sitting,26,1222
299998,2019-01-12 01:41:04.040,-0.902229,0.015485,-0.432351,-0.277919,0.014813,0.964061,7,S015,sitting,26,1222


In [15]:
# Number the rows in each  in window_size variable 
window_size = data.groupby(["block", "window"])["label"].transform("size")
data.groupby(["block", "window"]).ngroups # from inspection 8483.46 seconds (2:21:23.46)

8383

In [16]:
# Only keep windows with 50 rows
data = data[window_size == 50]
data.groupby(["block", "window"]).ngroups 

8347

In [17]:
# Give each window a unique numbers to make them independepent of blocks
data["window_n"] = data.groupby(["block", "window"]).ngroup()
data.tail()

,timestamp,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label,subject,activity,block,window,window_n
418379,2019-01-12 02:21:23.220,-1.005246,0.005829,-0.004372,-0.948225,0.009050,-0.311183,6,S015,standing,36,238,8346
418380,2019-01-12 02:21:23.240,-1.003755,0.018547,-0.013338,-0.949249,0.013714,-0.305369,6,S015,standing,36,238,8346
418381,2019-01-12 02:21:23.260,-0.988104,0.015914,-0.012909,-0.948196,0.010036,-0.314898,6,S015,standing,36,238,8346
418382,2019-01-12 02:21:23.280,-0.991594,0.001809,-0.013452,-0.949282,0.013173,-0.323505,6,S015,standing,36,238,8346
418383,2019-01-12 02:21:23.300,-0.981020,0.002347,-0.012963,-0.948158,0.010827,-0.318665,6,S015,standing,36,238,8346


#### Check how many windows contain more than one activity & drop windows that contain more than 1 activity

In [18]:
activities_per_window = data.groupby("window_n")["activity"].nunique()
print((activities_per_window == 1).sum())
print((activities_per_window > 1).sum())

7957
390


In [19]:
# Drop all windows that contain more than one activity
one_activity_per_window = activities_per_window[activities_per_window == 1].index
one_activity_per_window
data = data[data["window_n"].isin(one_activity_per_window)]
data.groupby("window_n").ngroups

7957

#### Calculate Features
- vector magnitude of each sensor as a column
- mean, sd and range

In [20]:
# Add vector magnitude columns
data["back_mag"] = np.sqrt(data["back_x"]**2 + data["back_y"]**2 + data["back_z"]**2)
data["thigh_mag"] = np.sqrt(data["thigh_x"]**2 + data["thigh_y"]**2 + data["thigh_z"]**2)
data.head()

,timestamp,back_x,back_y,back_z,thigh_x,thigh_y,thigh_z,label,subject,activity,block,window,window_n,back_mag,thigh_mag
0,2019-01-12 00:00:00.000,-0.999492,0.023192,0.025461,-0.935965,0.023979,-0.355521,6,S015,standing,0,0,0,1.000085,1.001499
1,2019-01-12 00:00:00.020,-0.995551,0.021414,0.042297,-0.928875,0.029540,-0.371151,6,S015,standing,0,0,0,0.996680,1.000717
2,2019-01-12 00:00:00.040,-0.989507,0.012187,0.038830,-0.942907,0.019677,-0.359649,6,S015,standing,0,0,0,0.990344,1.009360
3,2019-01-12 00:00:00.060,-0.991327,0.015926,0.056181,-0.942363,0.009682,-0.348916,6,S015,standing,0,0,0,0.993046,1.004930
4,2019-01-12 00:00:00.080,-0.997368,0.014239,0.035168,-0.930852,0.012002,-0.357421,6,S015,standing,0,0,0,0.998089,0.997185


In [21]:
# Group together the feature columns and windows to get the mean, sd & range
feature_cols = sensor_cols + ["back_mag", "thigh_mag"] 
group = data.groupby("window_n")[feature_cols]

means = group.mean().add_suffix("_mean")
sds = group.std().add_suffix("_sd")
ranges = (group.max() - group.min()).add_suffix("_range")

windows = pd.concat([means,sds,ranges], axis=1)
windows

,back_x_mean,back_y_mean,back_z_mean,thigh_x_mean,thigh_y_mean,thigh_z_mean,back_mag_mean,thigh_mag_mean,back_x_sd,back_y_sd,...,back_mag_sd,thigh_mag_sd,back_x_range,back_y_range,back_z_range,thigh_x_range,thigh_y_range,thigh_z_range,back_mag_range,thigh_mag_range
window_n,,,,,,,,,,,,,,,,,,,,,
0,-0.997566,-0.003537,0.039032,-0.926006,0.011276,-0.377946,0.998534,1.000764,0.009955,0.016718,...,0.009960,0.011793,0.045888,0.074017,0.041844,0.060773,0.131926,0.080139,0.045492,0.058954
1,-0.999350,0.003114,0.024276,-0.925395,0.012690,-0.376133,0.999868,0.999406,0.006835,0.017959,...,0.006805,0.011722,0.027156,0.071274,0.053030,0.045918,0.102795,0.081350,0.027757,0.049682
2,-0.998856,0.004546,0.041974,-0.939366,0.073732,-0.321426,1.000470,0.998528,0.028075,0.030832,...,0.028210,0.028966,0.161051,0.127368,0.099691,0.153978,0.288017,0.153493,0.161787,0.134443
4,-0.997292,-0.017871,0.035095,-0.950901,-0.028742,-0.267222,1.000832,1.006213,0.026677,0.063864,...,0.027112,0.123596,0.129428,0.254126,0.184376,0.728256,0.953008,0.616860,0.141814,0.677173
5,-0.997449,0.021474,0.021247,-0.931938,-0.018601,-0.358324,0.998340,0.999657,0.012523,0.022132,...,0.012713,0.025721,0.061842,0.142790,0.087011,0.138751,0.115028,0.194896,0.063451,0.147386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8342,-0.993424,0.003580,-0.011400,-0.946148,0.014241,-0.326608,0.993533,1.001138,0.006054,0.003421,...,0.006060,0.004504,0.021985,0.016042,0.036708,0.021135,0.021041,0.055657,0.021950,0.025264
8343,-0.994437,0.003214,-0.012957,-0.949592,0.011779,-0.312637,0.994595,0.999915,0.006272,0.005751,...,0.006306,0.004628,0.019579,0.032979,0.050731,0.038296,0.045153,0.051580,0.019943,0.025200
8344,-0.994123,0.006501,-0.009617,-0.948755,0.007271,-0.316775,0.994222,1.000317,0.005366,0.005822,...,0.005359,0.002193,0.022389,0.021726,0.021236,0.001883,0.034274,0.026733,0.022521,0.008062


In [22]:
# Add the activity type and subject to the df
windows["activity"] = data.groupby("window_n")["activity"].first() # Take the first activity of each window 
windows["subject"] = "S015"
windows.head()

,back_x_mean,back_y_mean,back_z_mean,thigh_x_mean,thigh_y_mean,thigh_z_mean,back_mag_mean,thigh_mag_mean,back_x_sd,back_y_sd,...,back_x_range,back_y_range,back_z_range,thigh_x_range,thigh_y_range,thigh_z_range,back_mag_range,thigh_mag_range,activity,subject
window_n,,,,,,,,,,,,,,,,,,,,,
0,-0.997566,-0.003537,0.039032,-0.926006,0.011276,-0.377946,0.998534,1.000764,0.009955,0.016718,...,0.045888,0.074017,0.041844,0.060773,0.131926,0.080139,0.045492,0.058954,standing,S015
1,-0.999350,0.003114,0.024276,-0.925395,0.012690,-0.376133,0.999868,0.999406,0.006835,0.017959,...,0.027156,0.071274,0.053030,0.045918,0.102795,0.081350,0.027757,0.049682,standing,S015
2,-0.998856,0.004546,0.041974,-0.939366,0.073732,-0.321426,1.000470,0.998528,0.028075,0.030832,...,0.161051,0.127368,0.099691,0.153978,0.288017,0.153493,0.161787,0.134443,standing,S015
4,-0.997292,-0.017871,0.035095,-0.950901,-0.028742,-0.267222,1.000832,1.006213,0.026677,0.063864,...,0.129428,0.254126,0.184376,0.728256,0.953008,0.616860,0.141814,0.677173,shuffling,S015
5,-0.997449,0.021474,0.021247,-0.931938,-0.018601,-0.358324,0.998340,0.999657,0.012523,0.022132,...,0.061842,0.142790,0.087011,0.138751,0.115028,0.194896,0.063451,0.147386,shuffling,S015


In [23]:
# Checks
print(windows.isna().sum())
windows["activity"].value_counts()

back_x_mean        0
back_y_mean        0
back_z_mean        0
thigh_x_mean       0
thigh_y_mean       0
thigh_z_mean       0
back_mag_mean      0
thigh_mag_mean     0
back_x_sd          0
back_y_sd          0
back_z_sd          0
thigh_x_sd         0
thigh_y_sd         0
thigh_z_sd         0
back_mag_sd        0
thigh_mag_sd       0
back_x_range       0
back_y_range       0
back_z_range       0
thigh_x_range      0
thigh_y_range      0
thigh_z_range      0
back_mag_range     0
thigh_mag_range    0
activity           0
subject            0
dtype: int64


activity
sitting      3896
lying        1261
standing     1241
walking       987
stairs        220
cycling       180
shuffling     116
running        56
Name: count, dtype: int64

#### Testing Functions to clean all partcipants data

In [76]:
# Import clean_participant and create_windows functions
from src.preprocessing import clean_participant, create_windows, create_features

In [25]:
s015 = clean_participant(data_directory / "S015.csv")

In [29]:
s015.info()

<class 'pandas.DataFrame'>
RangeIndex: 418392 entries, 0 to 418391
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  418392 non-null  datetime64[us]
 1   back_x     418392 non-null  float64       
 2   back_y     418392 non-null  float64       
 3   back_z     418392 non-null  float64       
 4   thigh_x    418392 non-null  float64       
 5   thigh_y    418392 non-null  float64       
 6   thigh_z    418392 non-null  float64       
 7   label      418392 non-null  int64         
 8   subject    418392 non-null  str           
 9   activity   418392 non-null  str           
dtypes: datetime64[us](1), float64(6), int64(1), str(2)
memory usage: 31.9 MB


In [52]:
s006 = clean_participant(data_directory / "S006.csv")

In [53]:
s006.info()
s006["timestamp"].diff().median()
s006["activity"].value_counts() * 0.02 / 60

<class 'pandas.DataFrame'>
RangeIndex: 198710 entries, 0 to 198709
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  198710 non-null  datetime64[us]
 1   back_x     198710 non-null  float64       
 2   back_y     198710 non-null  float64       
 3   back_z     198710 non-null  float64       
 4   thigh_x    198710 non-null  float64       
 5   thigh_y    198710 non-null  float64       
 6   thigh_z    198710 non-null  float64       
 7   label      198710 non-null  int64         
 8   subject    198710 non-null  str           
 9   activity   198710 non-null  str           
dtypes: datetime64[us](1), float64(6), int64(1), str(2)
memory usage: 15.2 MB


activity
sitting      42.171667
standing     10.446667
cycling       4.695000
walking       4.148667
lying         2.172667
shuffling     2.115333
stairs        0.486667
Name: count, dtype: float64

In [58]:
s015_window = create_windows(s015)

In [62]:
s015_window.nunique()
#s015_window.head()

timestamp    397850
back_x       397850
back_y       397850
back_z       397850
thigh_x      397850
thigh_y      397850
thigh_z      397850
label            10
subject           1
activity          8
block            37
window         1649
window_n       7957
dtype: int64

In [85]:
s015_features = create_features(s015_window)
s015_features.isna().sum()
#s015_features.info()

back_x_mean        0
back_y_mean        0
back_z_mean        0
thigh_x_mean       0
thigh_y_mean       0
thigh_z_mean       0
back_mag_mean      0
thigh_mag_mean     0
back_x_sd          0
back_y_sd          0
back_z_sd          0
thigh_x_sd         0
thigh_y_sd         0
thigh_z_sd         0
back_mag_sd        0
thigh_mag_sd       0
back_x_range       0
back_y_range       0
back_z_range       0
thigh_x_range      0
thigh_y_range      0
thigh_z_range      0
back_mag_range     0
thigh_mag_range    0
activity           0
subject            0
dtype: int64